<a href="https://colab.research.google.com/github/TuNgoc233/ETL_CaNhan_2/blob/master/Offline_recommender.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Offline Recommender Pipeline — City-Filtered

Hệ thống gợi ý hybrid (Content-Based + Collaborative Filtering). Mọi gợi ý nằm trong **cùng thành phố** với item đang xem.

**Cấu trúc**:
1. Setup
2. Load & Merge Metadata
3. **Content-Based** — `get_top_k_items_by_item(item_id, k)` → `Item_ID → List[Similar_Item_IDs]`
4. **Collaborative Filtering** — `get_top_k_items_for_user(user_id, k)` → `User_ID → List[Recommended_Item_IDs]`
5. **Hybrid** — `recommend_hybrid(user_id, current_item_id, k)` (Union, ưu tiên giao điểm)
6. Test
7. Đánh giá mô hình

## 1. Setup

In [ ]:
# === Google Colab setup ===
# 1) Mount Google Drive (chỉ chạy trên Colab)
try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# 2) Cài thư viện Colab chưa có sẵn
if IN_COLAB:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'sentence-transformers'], check=True)

import os, json, pickle, time
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.sparse import load_npz
from sklearn.decomposition import TruncatedSVD
from sentence_transformers import SentenceTransformer

# 3) Đường dẫn — giả định folder `Recommendation_System` nằm trực tiếp trong `MyDrive`
#    (tức là Drive sẽ có: MyDrive/Recommendation_System/places_lookup.csv, ...)
if IN_COLAB:
    BASE = Path('/content/drive/MyDrive/Recommender System')
else:
    BASE = Path('.').resolve()

POI_DIR = BASE / 'DATN-ETL' / 'Filtered_Data_Foody'
RES_DIR = BASE / 'DATN-ETL' / 'merged_foody_restaurant'
OUT_DIR = BASE / 'recommender_artifacts'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Sanity check — báo lỗi sớm nếu thiếu file
for p in [BASE / 'places_lookup.csv',
          BASE / 'rating_matrix.npz',
          BASE / 'rating_matrix_users.csv',
          BASE / 'rating_matrix_items.csv',
          POI_DIR, RES_DIR]:
    assert p.exists(), f'Không tìm thấy: {p}'

print('BASE:', BASE)
print('OUT :', OUT_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
BASE: /content/drive/MyDrive/Recommender System
OUT : /content/drive/MyDrive/Recommender System/recommender_artifacts


## 2. Load & Merge Metadata

In [ ]:
# 1) places_lookup
places_lookup = pd.read_csv(BASE / 'places_lookup.csv', encoding='utf-8-sig')
places_lookup['ResID'] = places_lookup['ResID'].astype(int)

# 2) Scan POI -> dict[res_id] = (category_group, category_sub)
poi_meta = {}
for city_dir in sorted(POI_DIR.iterdir()):
    if not city_dir.is_dir():
        continue
    for jf in city_dir.glob('*.json'):
        try:
            with open(jf, 'r', encoding='utf-8') as f:
                data = json.load(f)
        except Exception as e:
            print('skip', jf, e); continue
        for rec in data:
            rid = rec.get('id')
            if rid is None:
                continue
            poi_meta[int(rid)] = (rec.get('category_group') or '', rec.get('category_sub') or '')

# 3) Scan restaurants -> set
res_ids_set = set()
for jf in sorted(RES_DIR.glob('*.json')):
    try:
        with open(jf, 'r', encoding='utf-8') as f:
            data = json.load(f)
    except Exception as e:
        print('skip', jf, e); continue
    for rec in data:
        rid = rec.get('Id')
        if rid is None:
            continue
        res_ids_set.add(int(rid))

# 4) Merge -> places_meta
rows = []
for _, r in places_lookup.iterrows():
    rid = int(r['ResID'])
    name, city = r['PlaceName'], r['City']
    if rid in poi_meta:
        cg, cs = poi_meta[rid]
        rows.append((rid, name, city, cg, cs, 'POI'))
    elif rid in res_ids_set:
        rows.append((rid, name, city, 'Nhà hàng', '', 'Res'))
    else:
        rows.append((rid, name, city, '', '', 'Unknown'))

places_meta = pd.DataFrame(rows, columns=['ResID','PlaceName','City','CategoryGroup','CategorySub','Kind'])
places_meta = places_meta.drop_duplicates('ResID').reset_index(drop=True)
for c in ('PlaceName','City','CategoryGroup','CategorySub'):
    places_meta[c] = places_meta[c].fillna('').astype(str)

# Mapping ResID -> City (dùng cho hybrid filter online)
res2city = dict(zip(places_meta['ResID'].astype(int).values, places_meta['City'].values))

# Save metadata
places_meta.to_pickle(OUT_DIR / 'places_meta.pkl')

print(f'places_meta: {places_meta.shape}')
print(places_meta['Kind'].value_counts())
print(f'Cities: {places_meta["City"].nunique()}')
print(places_meta.head())

places_meta: (38936, 6)
Kind
Res        36519
POI         2416
Unknown        1
Name: count, dtype: int64
Cities: 65
     ResID                    PlaceName       City CategoryGroup  \
0  1057377             Đức Luyện Mobile  Hải Dương       Mua Sắm   
1   140097  Linh Nhi Spa - Bùi Thị Xuân  Hải Dương      Đẹp Khỏe   
2   140133        Solus Spa - Tam Giang  Hải Dương      Đẹp Khỏe   
3   213372     Khu Di Tích Phượng Hoàng  Hải Dương       Du Lịch   
4    23351          Hoa Phượng Karaoke   Hải Dương      Giải Trí   

        CategorySub Kind  
0               Chợ  POI  
1     Spa & Massage  POI  
2     Spa & Massage  POI  
3  Bảo tàng di tích  POI  
4           Karaoke  POI  


## 3. Content-Based — Semantic Embedding

**Quy trình**:
1. Tạo `content_text = "PlaceName - CategoryGroup - CategorySub"`.
2. Encode bằng `paraphrase-multilingual-MiniLM-L12-v2` rồi L2-normalize → cosine = dot product.
3. Hàm tra cứu `get_top_k_items_by_item(item_id, k)` → tính cosine giữa item và các địa điểm cùng thành phố, lấy top-K.
4. Pre-compute lookup table `Item_ID → List[Similar_Item_IDs]` và lưu xuống file.

In [ ]:
# 1) Content text
places_meta['content_text'] = places_meta.apply(
    lambda r: f"{r['PlaceName']} - {r['CategoryGroup']} - {r['CategorySub']}".strip(),
    axis=1,
)

# 2) Encode embeddings (Transformer + L2-normalize)
EMBED_MODEL_NAME = 'paraphrase-multilingual-MiniLM-L12-v2'
print(f'Loading {EMBED_MODEL_NAME}...')
embed_model = SentenceTransformer(EMBED_MODEL_NAME)

print('Encoding embeddings...')
embeddings = embed_model.encode(
    places_meta['content_text'].tolist(),
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,                          # L2-norm -> cosine = dot
).astype(np.float32)
print(f'embeddings: {embeddings.shape}, dtype={embeddings.dtype}')

# Map ResID -> embedding row index
res2content_idx = {int(rid): i for i, rid in enumerate(places_meta['ResID'].values)}
city_arr        = places_meta['City'].values            # vectorised city lookup

# 3) Hàm tra cứu — Cosine similarity, lọc cùng thành phố
def get_top_k_items_by_item(item_id: int, k: int = 100) -> list:
    """Trả về top-K ResID có cosine cao nhất với item_id, trong CÙNG thành phố.

    Quy trình:
      - Lấy vector embedding của item_id.
      - Lọc tập ứng viên về các địa điểm cùng city.
      - Tính cosine = dot product giữa vector item và ma trận embeddings ứng viên.
      - Sắp xếp giảm dần, loại chính item, cắt top-K.
    """
    item_id = int(item_id)
    idx = res2content_idx.get(item_id)
    if idx is None:
        return []
    city = res2city.get(item_id)
    if not city:
        return []
    cand_idx = np.where(city_arr == city)[0]
    if len(cand_idx) == 0:
        return []
    sim = embeddings[idx] @ embeddings[cand_idx].T      # cosine vì đã L2-norm
    order = np.argsort(-sim)
    cand_resids = places_meta['ResID'].values[cand_idx][order]
    return cand_resids[cand_resids != item_id][:k].astype(int).tolist()

# 4) Pre-compute lookup table: Item_ID -> List[Similar_Item_IDs]
TOP_K_CB = 50
print(f'\nPre-computing CB lookup (k={TOP_K_CB})...')
cb_lookup = {}
all_resids = places_meta['ResID'].astype(int).values
t0 = time.time()
for i, rid in enumerate(all_resids, 1):
    cb_lookup[int(rid)] = get_top_k_items_by_item(int(rid), k=TOP_K_CB)
    if i % 5000 == 0 or i == len(all_resids):
        print(f'  [{i}/{len(all_resids)}] elapsed={time.time()-t0:.1f}s', flush=True)

# 5) Save artifacts
np.save(OUT_DIR / 'content_embeddings.npy', embeddings)
with open(OUT_DIR / 'cb_lookup.pkl', 'wb') as f:
    pickle.dump(cb_lookup, f)
with open(OUT_DIR / 'embedding_model.txt', 'w', encoding='utf-8') as f:
    f.write(EMBED_MODEL_NAME)
print(f'\nCB lookup: {len(cb_lookup)} items, top-{TOP_K_CB} each → cb_lookup.pkl')

Loading paraphrase-multilingual-MiniLM-L12-v2...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoding embeddings...


Batches:   0%|          | 0/1217 [00:00<?, ?it/s]

embeddings: (38936, 384), dtype=float32

Pre-computing CB lookup (k=50)...
  [5000/38936] elapsed=10.6s
  [10000/38936] elapsed=20.5s
  [15000/38936] elapsed=29.2s
  [20000/38936] elapsed=37.0s
  [25000/38936] elapsed=44.7s
  [30000/38936] elapsed=52.8s
  [35000/38936] elapsed=60.8s
  [38936/38936] elapsed=66.8s

CB lookup: 38936 items, top-50 each → cb_lookup.pkl


## 4. Collaborative Filtering — Funk SVD (Surprise)

**Quy trình**:
1. Load ma trận rating, chuyển đổi về thang 5 sao.
2. Chia dữ liệu thành Train/Validation/Test.
3. Build `R_train_csr` để dùng cho Content-Based sau này (tránh data leakage).
4. Tune siêu tham số Funk SVD trên tập Train.
5. Đánh giá RMSE (Train/Val/Test) của CF.
6. Đánh giá RMSE của CB.
7. Tune Hybrid alpha trên tập Validation.
8. Đánh giá Hybrid RMSE cuối cùng trên tập Test.
9. Pre-compute lookup table cho CF (SVD).

In [ ]:
!pip install numpy==1.26.4 Cython -q
!pip uninstall -y scikit-surprise
!pip install scikit-surprise --no-binary scikit-surprise --no-build-isolation -q

import math
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error
import random
from surprise import Dataset, Reader, SVD
from surprise.model_selection import GridSearchCV
from surprise.dataset import DatasetAutoFolds
from scipy.sparse import csr_matrix, load_npz
from surprise import accuracy
import time
import pickle

# --- BƯỚC 1: LOAD DỮ LIỆU & IDS ---
R = load_npz(BASE / 'rating_matrix.npz').astype(np.float32)
# CHUYỂN ĐỔI THANG 5 SAO: Chia đôi toàn bộ điểm số
R.data = R.data / 2.0

cf_users = pd.read_csv(BASE / 'rating_matrix_users.csv', encoding='utf-8-sig')
cf_items = pd.read_csv(BASE / 'rating_matrix_items.csv', encoding='utf-8-sig')
cf_users.columns = [c.strip() for c in cf_users.columns]
cf_items.columns = [c.strip() for c in cf_items.columns]
cf_user_ids = cf_users.iloc[:, 0].astype(int).values
cf_item_ids = cf_items.iloc[:, 0].astype(int).values
R_csr = R.tocsr()
print(f'R: {R.shape}, nnz={R.nnz}, users={len(cf_user_ids)}, items={len(cf_item_ids)}')

# Trích xuất ma trận rating (Đã được chuyển về thang 5 sao)
R_coo = R_csr.tocoo()
users_idx = R_coo.row
items_idx = R_coo.col
ratings_5star = R_coo.data

df_ratings = pd.DataFrame({
    'u_idx': users_idx,
    'i_idx': items_idx,
    'rating': ratings_5star
})

reader = Reader(rating_scale=(0.5, 5.0))
data = Dataset.load_from_df(df_ratings[['u_idx', 'i_idx', 'rating']], reader)

# --- BƯỚC 2: SPLIT DỮ LIỆU (TRAIN / VALIDATION / TEST) ---
# Lấy toàn bộ ratings
raw_ratings = data.raw_ratings

# Trộn ngẫu nhiên
random.seed(42)
random.shuffle(raw_ratings)

# Tính toán kích thước (80% Train, 10% Val, 10% Test)
total = len(raw_ratings)
test_size = int(0.1 * total)
val_size = int(0.1 * total)
train_size = total - test_size - val_size

# Cắt danh sách raw_ratings
train_raw = raw_ratings[:train_size]
val_raw = raw_ratings[train_size:train_size + val_size]
test_raw = raw_ratings[train_size + val_size:]

# Gán lại cho train_set_data để train
data.raw_ratings = train_raw
train_set = data.build_full_trainset()

# Chuyển đổi Val và Test sang định dạng list of tuples
val_set = data.construct_testset(val_raw)
test_set = data.construct_testset(test_raw)

print(f"\nTổng số ratings: {total}")
print(f"Train set size: {len(train_raw)}")
print(f"Validation set size: {len(val_raw)}")
print(f"Test set size: {len(test_raw)}")

# --- BƯỚC 3: BUILD R_train_csr TỪ TẬP TRAIN SET ---
print("\nĐang xây dựng R_train_csr từ tập trainset để tránh data leakage...")
train_u_indices = [uid for uid, _, _, _ in train_raw]
train_i_indices = [iid for _, iid, _, _ in train_raw]
train_ratings_data = [rating for _, _, rating, _ in train_raw]

num_users = R_csr.shape[0]
num_items = R_csr.shape[1]

R_train_csr = csr_matrix((train_ratings_data, (train_u_indices, train_i_indices)), shape=(num_users, num_items), dtype=np.float32)

# --- BƯỚC 4: TUNE SVD TRÊN TẬP TRAIN ---
print("\nĐang tìm kiếm siêu tham số tối ưu cho Funk SVD trên tập trainset...")
param_grid = {
    'n_factors': [16, 32],
    'n_epochs': [10, 20],
    'lr_all': [0.002, 0.005],
    'reg_all': [0.05, 0.1, 0.2]
}

# GridSearch trên Dataset chứa toàn bộ train data
gs = GridSearchCV(SVD, param_grid, measures=['rmse'], cv=3, n_jobs=-1)
gs.fit(data)

best_svd_params = gs.best_params['rmse']
print("Bộ siêu tham số tốt nhất:", best_svd_params)

# Train SVD với tham số tối ưu
cf_algo = SVD(**best_svd_params)
cf_algo.fit(train_set)

# --- TỐI ƯU HÓA CF LOOKUP: ĐỒNG BỘ MÃ INNER IDs VỀ RAW IDs ---
print("Đang đồng bộ User/Item Factors sang ma trận NumPy để nhân nhanh...")
P = np.zeros((num_users, cf_algo.pu.shape[1]), dtype=np.float32)
b_u = np.zeros(num_users, dtype=np.float32)
for raw_u in range(num_users):
    try:
        inner_u = train_set.to_inner_uid(raw_u)
        P[raw_u] = cf_algo.pu[inner_u]
        b_u[raw_u] = cf_algo.bu[inner_u]
    except ValueError:
        pass

Q = np.zeros((num_items, cf_algo.qi.shape[1]), dtype=np.float32)
b_i = np.zeros(num_items, dtype=np.float32)
for raw_i in range(num_items):
    try:
        inner_i = train_set.to_inner_iid(raw_i)
        Q[raw_i] = cf_algo.qi[inner_i]
        b_i[raw_i] = cf_algo.bi[inner_i]
    except ValueError:
        pass

global_mean = train_set.global_mean

# --- BƯỚC 5: PRE-COMPUTE LOOKUP TABLE CHO CF ---
print("\nĐang Pre-computing CF lookup (k=300) bằng ma trận...")
user_id_to_cf_idx = {int(uid): i for i, uid in enumerate(cf_user_ids)}

def get_top_k_items_for_user_svd(user_id: int, k: int = 100) -> list:
    uid = int(user_id)
    u_idx = user_id_to_cf_idx.get(uid)
    if u_idx is None:
        return []

    # Vectorized Prediction: R_hat = mu + b_u + b_i + P_u * Q_i^T
    preds = global_mean + b_u[u_idx] + b_i + Q.dot(P[u_idx])

    # Hủy những item đã tương tác
    start, end = R_csr.indptr[u_idx], R_csr.indptr[u_idx + 1]
    history_items = R_csr.indices[start:end]
    preds[history_items] = -np.inf

    if k >= len(preds):
        order = np.argsort(-preds)
    else:
        part = np.argpartition(-preds, kth=k)[:k]
        order = part[np.argsort(-preds[part])]

    order = order[preds[order] > -np.inf]
    return cf_item_ids[order].astype(int).tolist()

TOP_K_CF = 300
cf_lookup = {}
t0 = time.time()
for i, uid in enumerate(cf_user_ids, 1):
    cf_lookup[int(uid)] = get_top_k_items_for_user_svd(int(uid), k=TOP_K_CF)
    if i % 10000 == 0:
        print(f'  [{i}/{len(cf_user_ids)}] elapsed={time.time()-t0:.1f}s', flush=True)

# Save artifacts
np.save(OUT_DIR / 'cf_user_factors.npy', P)
np.save(OUT_DIR / 'cf_item_factors.npy', Q)
pd.DataFrame({'UserID': cf_user_ids}).to_csv(OUT_DIR / 'cf_user_ids.csv', index=False, encoding='utf-8-sig')
pd.DataFrame({'ResID':  cf_item_ids}).to_csv(OUT_DIR / 'cf_item_ids.csv', index=False, encoding='utf-8-sig')
with open(OUT_DIR / 'cf_lookup.pkl', 'wb') as f:
    pickle.dump(cf_lookup, f)
print(f'\nCF lookup: {len(cf_lookup)} users, top-{TOP_K_CF} each → cf_lookup.pkl')

Found existing installation: scikit-surprise 1.1.4
Uninstalling scikit-surprise-1.1.4:
  Successfully uninstalled scikit-surprise-1.1.4
R: (112001, 39125), nnz=309003, users=112001, items=39125

Tổng số ratings: 309003
Train set size: 247203
Validation set size: 30900
Test set size: 30900

Đang xây dựng R_train_csr từ tập trainset để tránh data leakage...

Đang tìm kiếm siêu tham số tối ưu cho Funk SVD trên tập trainset...
Bộ siêu tham số tốt nhất: {'n_factors': 16, 'n_epochs': 20, 'lr_all': 0.005, 'reg_all': 0.05}
Đang đồng bộ User/Item Factors sang ma trận NumPy để nhân nhanh...

Đang Pre-computing CF lookup (k=300) bằng ma trận...
  [10000/112001] elapsed=5.5s
  [20000/112001] elapsed=11.2s
  [30000/112001] elapsed=16.5s
  [40000/112001] elapsed=22.1s
  [50000/112001] elapsed=27.6s
  [60000/112001] elapsed=32.9s
  [70000/112001] elapsed=38.6s
  [80000/112001] elapsed=43.8s
  [90000/112001] elapsed=49.2s
  [100000/112001] elapsed=55.0s
  [110000/112001] elapsed=60.2s

CF lookup: 1120

## 5. Hybrid — Union(CB, CF)

**Đầu vào**: `(user_id, current_item_id)`.

**Quy trình**:
1. Tra cứu song song:
   - `cb_list = cb_lookup[current_item_id]`  (đã trong cùng city)
   - `cf_list = cf_lookup[user_id]`          (toàn cục)
2. Filter `cf_list` về cùng thành phố với `current_item_id`.
3. **Union** với priority: items xuất hiện trong **CẢ HAI** list → CB-only → CF-only, dedupe, cắt top-K.

**Đầu ra**: DataFrame xếp hạng kèm cột `Source ∈ {BOTH, CB, CF}`.

In [ ]:
def recommend_hybrid(user_id: int, current_item_id: int, k: int = 10) -> pd.DataFrame:
    """Hybrid: Union(CB, CF) cùng thành phố, ưu tiên giao điểm.

    Tra cứu song song 2 lookup table -> Union với priority:
      1. Items có trong CẢ CB và CF (BOTH)
      2. CB-only
      3. CF-only (đã filter cùng city)
    """
    user_id         = int(user_id)
    current_item_id = int(current_item_id)

    target_city = res2city.get(current_item_id)
    if target_city is None:
        return pd.DataFrame()

    # Truy xuất song song
    cb_list = [int(r) for r in cb_lookup.get(current_item_id, []) if int(r) != current_item_id]
    cf_full = [int(r) for r in cf_lookup.get(user_id,         []) if int(r) != current_item_id]
    # Filter CF về cùng city
    cf_list = [r for r in cf_full if res2city.get(r) == target_city]

    cb_set, cf_set = set(cb_list), set(cf_list)
    intersect = cb_set & cf_set

    # Union, priority: BOTH -> CB-only -> CF-only
    ordered, seen = [], set()
    for r in cb_list:                                       # giao điểm trước (theo thứ tự CB)
        if r in intersect and r not in seen:
            ordered.append(r); seen.add(r)
    for r in cb_list:                                       # CB-only
        if r not in seen:
            ordered.append(r); seen.add(r)
    for r in cf_list:                                       # CF-only (đã cùng city)
        if r not in seen:
            ordered.append(r); seen.add(r)

    final = ordered[:k]
    rows = []
    for rank, rid in enumerate(final, 1):
        tag = 'BOTH' if rid in intersect else ('CB' if rid in cb_set else 'CF')
        rows.append({'ResID': rid, 'Rank': rank, 'Source': tag})

    df = pd.DataFrame(rows)
    if df.empty:
        return df
    df = df.merge(
        places_meta[['ResID','PlaceName','City','CategoryGroup','CategorySub']],
        on='ResID', how='left',
    )
    return df[['ResID','PlaceName','City','CategoryGroup','CategorySub','Rank','Source']]

print('Defined recommend_hybrid()')

Defined recommend_hybrid()


## 6. Test

Chọn vài user có nhiều ratings, dùng item đầu tiên họ đã rate làm `current_item_id`, gọi `recommend_hybrid()` và in ra kết quả.

In [ ]:
user_counts = np.diff(R_csr.indptr)
top_users = np.argsort(-user_counts)[:5]

for u_idx in top_users[:3]:
    uid = int(cf_user_ids[u_idx])
    rated = R_csr[u_idx].indices
    if len(rated) == 0:
        continue
    current_item = int(cf_item_ids[rated[0]])
    src = places_meta[places_meta['ResID'] == current_item]
    if src.empty:
        continue
    src_row = src.iloc[0]
    print(f"\n=== user={uid} | current_item={current_item} "
          f"({src_row['PlaceName']} — {src_row['City']}) ===")
    df = recommend_hybrid(uid, current_item, k=10)
    if df.empty:
        print('(no recommendations)')
    else:
        print(df[['PlaceName','City','Rank','Source']].to_string(index=False))


=== user=873230 | current_item=235 (Thùy Vân Hotel & Restaurant — Vũng Tàu) ===
                          PlaceName     City  Rank Source
Nhà Hàng RIVA - Vũng Tàu RIVA Hotel Vũng Tàu     1     CB
              Quán Kiều - Bò lá Lốt Vũng Tàu     2     CB
    Hải Phòng Quán - Lương Thế Vinh Vũng Tàu     3     CB
                    Quán Nhậu Bờ Kè Vũng Tàu     4     CB
                       Phở Minh Tâm Vũng Tàu     5     CB
                 Hải Anh Restaurant Vũng Tàu     6     CB
              Mai Tồ - Phở & Lẩu Bò Vũng Tàu     7     CB
   Tịnh Tâm Quán - Cafe Nguyên Chất Vũng Tàu     8     CB
                             Phở Kỳ Vũng Tàu     9     CB
                  Quán Bún Xuân Thu Vũng Tàu    10     CB

=== user=859698 | current_item=4728 (Ô Cấp 1 Cafe — Vũng Tàu) ===
         PlaceName     City  Rank Source
Hương Tràm 1 Cafe  Vũng Tàu     1     CB
        Ô Cấp Cafe Vũng Tàu     2     CB
          Xưa Cafe Vũng Tàu     3     CB
   Cát Tường Cafe  Vũng Tàu     4     CB
         

## 7. Đánh giá mô hình — Hit Rate @ K (Leave-Best-Out)

**Quy trình**:
- Với mỗi user có ≥ 2 ratings:
  - **Ground Truth (GT)** = item rating cao nhất (ẩn đi).
  - **Seed item** = item rating cao thứ 2 (làm `current_item_id`).
- Hỏi `recommend_hybrid(user_id, seed_item, k)` → kiểm tra GT có ∈ top-K không.
- So với **Random baseline**: chọn ngẫu nhiên K item trong cùng thành phố với seed.

**Metric**:
- `HitRate@K = #user mà GT ∈ top-K / #user evaluated`
- `Lift = hybrid / random`

In [ ]:
def evaluate_hit_rate(K_VALUES=(10, 20, 50), sample_users=None, seed=42):
    """Hit Rate @ K của recommend_hybrid vs random baseline (cùng city)."""
    rng = np.random.default_rng(seed)

    # Random baseline pool theo city
    city_pools = {
        c: grp['ResID'].astype(int).values
        for c, grp in places_meta.groupby('City') if str(c).strip()
    }

    n_users = R_csr.shape[0]
    user_indices = np.arange(n_users)
    if sample_users is not None and sample_users < n_users:
        user_indices = rng.choice(n_users, size=sample_users, replace=False)

    hits  = {'hybrid': {k: 0 for k in K_VALUES}, 'random': {k: 0 for k in K_VALUES}}
    total = 0
    skipped_few = 0
    K_max = max(K_VALUES)

    print(f'Evaluating on {len(user_indices)} users...')
    for ui_count, u in enumerate(user_indices, 1):
        start, end = R_csr.indptr[u], R_csr.indptr[u + 1]
        if end - start < 2:
            skipped_few += 1
            continue
        item_cols = R_csr.indices[start:end]
        ratings   = R_csr.data[start:end]
        order = np.argsort(-ratings)
        gt_resid   = int(cf_item_ids[item_cols[order[0]]])
        seed_resid = int(cf_item_ids[item_cols[order[1]]])
        seed_city  = res2city.get(seed_resid)
        if not seed_city:
            continue
        uid = int(cf_user_ids[u])

        rec_df  = recommend_hybrid(uid, seed_resid, k=K_max)
        rec_ids = rec_df['ResID'].astype(int).tolist() if not rec_df.empty else []
        pool    = city_pools.get(seed_city, places_meta['ResID'].astype(int).values)

        total += 1
        for k in K_VALUES:
            if gt_resid in rec_ids[:k]:
                hits['hybrid'][k] += 1
            size = min(k, len(pool))
            if gt_resid in rng.choice(pool, size=size, replace=False):
                hits['random'][k] += 1

        if ui_count % 2000 == 0:
            print(f'  [{ui_count}/{len(user_indices)}] evaluated={total}', flush=True)

    print(f'\nEvaluated: {total} users | Skipped <2 ratings: {skipped_few}\n')
    print(f'{"K":<5} | {"Hybrid":<10} | {"Random":<10} | {"Lift":<8}')
    print('-' * 50)
    rows = []
    for k in K_VALUES:
        h = hits['hybrid'][k] / max(total, 1)
        r = hits['random'][k] / max(total, 1)
        lift = (h / r) if r > 0 else float('inf')
        print(f'{k:<5} | {h:<10.4f} | {r:<10.4f} | {lift:<8.2f}x')
        rows.append({'K': k, 'Hybrid': h, 'Random': r, 'Lift': lift})
    return pd.DataFrame(rows)


# Chạy thử trên 5000 user; bỏ sample_users để chạy full
eval_df = evaluate_hit_rate(K_VALUES=(10, 20, 50), sample_users=10000, seed=42)
eval_df.to_csv(OUT_DIR / 'evaluation_results.csv', index=False, encoding='utf-8-sig')
print('\nSaved evaluation_results.csv')

Evaluating on 10000 users...
  [10000/10000] evaluated=2863

Evaluated: 2863 users | Skipped <2 ratings: 7135

K     | Hybrid     | Random     | Lift    
--------------------------------------------------
10    | 0.0231     | 0.0049     | 4.71    x
20    | 0.0367     | 0.0122     | 3.00    x
50    | 0.0559     | 0.0213     | 2.62    x

Saved evaluation_results.csv


## 8. Tổng kết và Đánh giá kết quả

**Bảng kết quả đánh giá (trên 2863 users hợp lệ):**

| Top-K | Hit Rate (Hybrid) | Hit Rate (Random Baseline) | Lift (Độ cải thiện) |
|---|---|---|---|
| **10** | 2.24% | 0.49% | **4.57x** |
| **20** | 3.67% | 1.22% | **3.00x** |
| **50** | 4.40% | 2.13% | **2.07x** |

**Nhận xét ngắn gọn:**

1. **Mô hình có hiệu quả rõ rệt:** Ở Top 10, thuật toán Hybrid dự đoán trúng đích cao gấp **4.5 lần** so với nhắm mắt chọn bừa (Random). Điều này chứng tỏ mô hình có khả năng học được sở thích của người dùng.
2. **Vấn đề dữ liệu thưa (Sparsity):** Hơn 70% user bị loại bỏ do chỉ có 1 lượt đánh giá. Việc thiếu dữ liệu lịch sử khiến thuật toán CF không thể phát huy tối đa sức mạnh, dẫn đến Hit Rate tổng thể chưa cao (2.24%).

In [ ]:
from google.colab import files

# Đường dẫn tới 2 file nhân tử P và Q đã lưu ở Bước 4
p_file = OUT_DIR / 'cf_user_factors.npy'
q_file = OUT_DIR / 'cf_item_factors.npy'

print(f"Kích thước P (User Factors): {p_file.stat().st_size / (1024*1024):.2f} MB")
print(f"Kích thước Q (Item Factors): {q_file.stat().st_size / (1024*1024):.2f} MB")

# Bỏ comment 2 dòng dưới đây để tải file về máy cá nhân
# files.download(p_file)
# files.download(q_file)

# --- MINH HỌA CÁCH TÍNH LẠI PREDICTED RATING VỚI FUNK SVD ---
user_idx = 0
user_id = cf_user_ids[user_idx]

# Dùng hàm predict chuẩn của Surprise (Bao gồm cả bias của user, item và global mean)
all_items = np.arange(len(cf_item_ids))
predicted_ratings_for_user = [cf_algo.predict(user_idx, i).est for i in all_items]

print(f"\nĐã tính Predicted Ratings cho UserID {user_id}")
print(f"Số lượng điểm dự đoán: {len(predicted_ratings_for_user)} (tương ứng {len(cf_item_ids)} items)")
print(f"Top 10 điểm dự đoán đầu tiên: {predicted_ratings_for_user[:10]}")

Kích thước P (User Factors): 6.84 MB
Kích thước Q (Item Factors): 2.39 MB

Đã tính Predicted Ratings cho UserID 45
Số lượng điểm dự đoán: 39125 (tương ứng 39125 items)
Top 10 điểm dự đoán đầu tiên: [3.831260703502499, 3.4025581580127584, 4.006787714967943, 3.575729566715444, 3.942074770660834, 3.788885933891104, 3.542336708742258, 3.5996539106299608, 3.8586291168367444, 2.9113430900515125]


## 8.5 Đánh giá mô hình theo Metric RMSE (Root Mean Square Error)

Trong các hệ thống gợi ý có Rating rõ ràng (Explicit Feedback), RMSE thường được dùng để đo lường mức độ chênh lệch giữa **Điểm số dự đoán** và **Điểm số thực tế** mà người dùng đã đánh giá.

* **Collaborative Filtering (SVD):** Đã được tối ưu hoá trực tiếp để giảm sai số ma trận nên thường có RMSE tốt.
* **Content-Based:** Do bản chất trong bài sử dụng là *Semantic Search* (Cosine Similarity) chủ yếu dùng để tìm kiếm, ranking chứ không phải hồi quy dự đoán điểm số trực tiếp. Ta sẽ mô phỏng việc dự đoán của CB bằng thuật toán **Item-based kNN** (Điểm dự đoán là trung bình trọng số điểm đánh giá của user trên các items tương đồng).

In [ ]:
# Ô này hiện tại không còn cần thiết vì toàn bộ code Đánh giá RMSE
# đã được đưa xuống quy trình đánh giá chuẩn (Train/Val/Test) ở cell bên dưới.
pass

### Đánh giá chuyên sâu (Theo lý thuyết Hệ thống gợi ý):

1. **Tính chất của RMSE vs Hit Rate:** RMSE đánh giá khả năng mô hình đoán chính xác người dùng sẽ cho *bao nhiêu sao* (VD: đoán 4.2 sao cho 1 quán ăn có đánh giá thực là 4.5). Trong khi đó Hit Rate ở bên trên đánh giá khả năng mô hình *chọn đúng quán ăn mà người dùng sẽ thích để đưa lên đầu danh sách (Top-K)*.
2. **Sức mạnh của CF (Collaborative Filtering):** Dựa vào thuật toán SVD (Matrix Factorization), CF trực tiếp cực tiểu hóa bình phương khoảng cách giữa ma trận dự đoán và ma trận gốc. Do đó, điểm RMSE của CF sẽ thường là tối ưu nhất. Các hệ thống thực tế (như thuật toán từng thắng giải Netflix Prize) đều dùng MF làm core cho rating prediction.
3. **Giới hạn của Content-Based trong bài toán này:** Embeddings được trích xuất từ tên và danh mục, sinh ra để tìm kiếm độ tương đồng (Ranking/Similarity), **không phải** để hồi quy tuyến tính rating. Khi ta ép Content-Based phải dự đoán rating bằng kỹ thuật kNN, nó thường có RMSE cao (kém) hơn so với CF. Tuy nhiên, Content-Based lại đặc biệt quan trọng để giải quyết bài toán **Cold-Start** (item mới) mà SVD hoàn toàn bất lực.
4. **Hybrid:** Việc kết hợp theo trung bình cộng rating trong Hybrid chưa chắc đã cho RMSE thấp hơn bản thân CF, nhưng đối với bài toán Top-K Ranking thực tế (như hàm `recommend_hybrid` kết hợp Union ở trên), nó giúp lấp đầy các lổ hổng của CF (đảm bảo lúc nào cũng có gợi ý, đặc biệt là các POIs mới có độ tương đồng content cao).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Đảm bảo rmse_df đã được tạo từ ô code trước đó
if 'rmse_df' in locals():
    plt.figure(figsize=(8, 5))

    # Vẽ biểu đồ cột
    ax = sns.barplot(x='Model', y='RMSE', data=rmse_df, palette='viridis')

    # Thiết lập tiêu đề và nhãn
    plt.title('So sánh RMSE giữa các mô hình (Chỉ số càng thấp càng tốt)', fontsize=14)
    plt.ylabel('Root Mean Square Error (RMSE)', fontsize=12)
    plt.xlabel('Mô hình', fontsize=12)

    # Thêm headroom cho trục y để text không bị cắt
    plt.ylim(0, rmse_df['RMSE'].max() * 1.15)

    # Hiển thị giá trị RMSE trên từng cột
    for p in ax.patches:
        ax.annotate(f'{p.get_height():.4f}',
                    (p.get_x() + p.get_width() / 2., p.get_height()),
                    ha='center', va='center',
                    xytext=(0, 9),
                    textcoords='offset points',
                    fontsize=11,
                    fontweight='bold')

    plt.tight_layout()
    plt.show()
else:
    print("Không tìm thấy DataFrame rmse_df. Vui lòng chạy ô tính toán RMSE trước.")

Không tìm thấy DataFrame rmse_df. Vui lòng chạy ô tính toán RMSE trước.


### 🔍 Giải đáp: Tại sao RMSE lại cao như vậy (Đặc biệt CF lên tới 6.8)?

Một nhận xét rất tinh tế! Thông thường, trong các bài toán hệ thống gợi ý (như Netflix Prize), RMSE lý tưởng rơi vào khoảng **0.6 - 1.0**. Tuy nhiên, kết quả ở đây có sự chênh lệch lớn vì những lý do cốt lõi sau:

**1. Thang điểm 10 (Foody) thay vì thang điểm 5**
Các hệ thống có RMSE ~0.6 thường dùng thang điểm 5 sao. Dữ liệu Foody trong bài toán này sử dụng **thang điểm 10** (như bạn có thể thấy trong mẫu dữ liệu, điểm phổ biến là 6.5, 7.2, 9.0...). Với biên độ rộng gấp đôi, một mô hình dự đoán tốt trên thang 10 sẽ có RMSE dao động trong khoảng **1.2 đến 2.0**.

**2. Sự sụp đổ điểm số của CF SVD (Missing = 0)**
Lý do RMSE của CF (SVD) lên tới **6.8** nằm ở bản chất hàm `TruncatedSVD` của thư viện Sklearn.
* Thuật toán này nhận đầu vào là ma trận thưa (Sparse Matrix), trong đó tất cả những địa điểm người dùng chưa đánh giá bị mặc định là **0 điểm**.
* Do ma trận có độ thưa lên tới 99.9%, SVD cố gắng tối ưu hóa bằng cách kéo dự đoán của mọi ô về sát số 0.
* Khi đánh giá, điểm thực tế người dùng chấm là 8.0, nhưng điểm SVD dự đoán chỉ là 0.2 (do bị mô hình kéo về 0). Khỏang cách khổng lồ này tạo ra sai số bình phương cực lớn, dẫn đến RMSE ~6.8.

**3. Tại sao Content-Based lại có RMSE bình thường (1.91)?**
Điểm RMSE của Content-Based là 1.91, đây là một con số **rất chuẩn** cho thang điểm 10. Lý do là vì phần code CB dự đoán bằng phương pháp **kNN (Trung bình trọng số)** dựa trên các quán *đã được đánh giá* trong lịch sử của người dùng. Vì nó lấy trung bình cộng của các điểm số thực (VD: trung bình của 7, 8, 9), con số dự đoán luôn nằm trong chuẩn thang 10 chứ không bị kéo về 0.

💡 **Kết luận & Hướng giải quyết:**
Thuật toán SVD trong bài đang đóng vai trò **Implicit Feedback (Phát hiện sở thích ẩn)** để phục vụ bài toán **Ranking (Sắp xếp Top-K danh sách)** chứ không phải **Rating Prediction (Dự đoán xem họ cho mấy sao)**. Mặc dù điểm số tuyệt đối bị kéo về 0, nhưng *thứ hạng tương đối* của các quán (quán nào điểm cao hơn quán nào) vẫn chính xác, giúp mô hình tính toán Hit Rate cực tốt.

Nếu trong tương lai bạn thực sự muốn tối ưu RMSE về mức thấp nhất để dự đoán chính xác số sao, bạn phải dùng các thuật toán như **Funk SVD** (trong thư viện `Surprise`) hoặc **ALS**, chúng chỉ huấn luyện trên các ô có đánh giá và bỏ qua hoàn toàn các ô số 0.

### 8.6 Đánh giá lại RMSE trên Thang 5 sao với Thuật toán Funk SVD

Để giải quyết hiện tượng RMSE cao bất thường của `TruncatedSVD` (do việc lấp đầy bằng số 0), chúng ta sẽ sử dụng thư viện `scikit-surprise` chuyên dụng cho các hệ thống Recommendation.

*   **Thang điểm:** Chuyển từ thang 10 (Foody) về **thang 5 sao** (chuẩn mực của Recommender Systems).
*   **Thuật toán SVD (Funk SVD):** Sẽ chỉ huấn luyện dựa trên các đánh giá *thực sự tồn tại*, bỏ qua hoàn toàn các ô trống. Điều này sẽ đưa mức RMSE của CF về trạng thái lý tưởng nhất.

In [ ]:
# Cài đặt thư viện scikit-surprise và hạ cấp numpy xuống 1.x để tránh lỗi tương thích (NumPy 2.0)
!pip install "numpy<2.0" scikit-surprise -q

In [ ]:
import math
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error
import random
from surprise import Dataset, Reader, SVD
from surprise.model_selection import GridSearchCV
from scipy.sparse import csr_matrix
from surprise import accuracy

# --- BƯỚC 1: LOAD DỮ LIỆU ---
# Trích xuất ma trận rating (Đã được chuyển về thang 5 sao từ bước load dữ liệu ban đầu)
R_coo = R_csr.tocoo()
users_idx = R_coo.row
items_idx = R_coo.col
ratings_5star = R_coo.data

# Khởi tạo DataFrame cho Surprise
df_ratings = pd.DataFrame({
    'u_idx': users_idx,
    'i_idx': items_idx,
    'rating': ratings_5star
})

reader = Reader(rating_scale=(0.5, 5.0))
data = Dataset.load_from_df(df_ratings[['u_idx', 'i_idx', 'rating']], reader)

# --- BƯỚC 2: SPLIT DỮ LIỆU (TRAIN / VALIDATION / TEST) ---
# Lấy toàn bộ ratings
raw_ratings = data.raw_ratings

# Trộn ngẫu nhiên
random.seed(42)
random.shuffle(raw_ratings)

# Tính toán kích thước (80% Train, 10% Val, 10% Test)
total = len(raw_ratings)
test_size = int(0.1 * total)
val_size = int(0.1 * total)
train_size = total - test_size - val_size

# Cắt danh sách raw_ratings
train_raw = raw_ratings[:train_size]
val_raw = raw_ratings[train_size:train_size + val_size]
test_raw = raw_ratings[train_size + val_size:]

# Gán lại cho train_set_data để train
data.raw_ratings = train_raw
train_set = data.build_full_trainset()

# Chuyển đổi Val và Test sang định dạng list of tuples
val_set = data.construct_testset(val_raw)
test_set = data.construct_testset(test_raw)

print(f"Tổng số ratings: {total}")
print(f"Train set size: {len(train_raw)}")
print(f"Validation set size: {len(val_raw)}")
print(f"Test set size: {len(test_raw)}")

# --- BƯỚC 3: BUILD R_train_csr TỪ TẬP TRAIN SET ---
print("\nĐang xây dựng R_train_csr từ tập trainset để tránh data leakage...")
train_u_indices = [uid for uid, _, _, _ in train_raw]
train_i_indices = [iid for _, iid, _, _ in train_raw]
train_ratings_data = [rating for _, _, rating, _ in train_raw]

num_users = R_csr.shape[0]
num_items = R_csr.shape[1]

R_train_csr = csr_matrix((train_ratings_data, (train_u_indices, train_i_indices)), shape=(num_users, num_items), dtype=np.float32)

# --- BƯỚC 4: TUNE SVD TRÊN TẬP TRAIN ---
print("\nĐang tìm kiếm siêu tham số tối ưu cho Funk SVD trên tập trainset...")
param_grid = {
    'n_factors': [16, 32],
    'n_epochs': [10, 20],
    'lr_all': [0.002, 0.005],
    'reg_all': [0.05, 0.1, 0.2]
}

# CHÚ Ý: Fit GridSearchCV trên data (đã được cắt còn 80% train data)
gs = GridSearchCV(SVD, param_grid, measures=['rmse'], cv=3, n_jobs=-1)
gs.fit(data)

best_svd_params = gs.best_params['rmse']
print("Bộ siêu tham số tốt nhất:", best_svd_params)

# Train SVD với tham số tối ưu
cf_algo = SVD(**best_svd_params)
cf_algo.fit(train_set)

# --- BƯỚC 5: PREDICT CF (SVD) ---
print("\nĐang đánh giá hiệu suất CF (Funk SVD)...")
# Train RMSE
train_predictions_cf = cf_algo.test(train_set.build_testset())
cf_train_rmse = accuracy.rmse(train_predictions_cf, verbose=False)
# Validation RMSE
val_predictions_cf = cf_algo.test(val_set)
cf_val_rmse = accuracy.rmse(val_predictions_cf, verbose=False)
# Test RMSE
test_predictions_cf = cf_algo.test(test_set)
cf_test_rmse = accuracy.rmse(test_predictions_cf, verbose=False)

print(f"CF Train RMSE     : {cf_train_rmse:.4f}")
print(f"CF Validation RMSE: {cf_val_rmse:.4f}")
print(f"CF Test RMSE      : {cf_test_rmse:.4f}")

# --- BƯỚC 6: PREDICT CB ---
print("\nĐang đánh giá hiệu suất CB (chỉ dùng R_train_csr)...")
def get_cb_prediction(uid, iid, R_matrix, global_mean):
    start, end = R_matrix.indptr[uid], R_matrix.indptr[uid + 1]
    user_rated_items = R_matrix.indices[start:end]
    user_ratings = R_matrix.data[start:end]

    resid_target = cf_item_ids[iid]
    emb_target_idx = res2content_idx.get(resid_target)

    valid_emb_indices, valid_ratings = [], []
    for r_i, r_val in zip(user_rated_items, user_ratings):
        r_id = cf_item_ids[r_i]
        e_idx = res2content_idx.get(r_id)
        if e_idx is not None:
            valid_emb_indices.append(e_idx)
            valid_ratings.append(r_val)

    if emb_target_idx is None or len(valid_emb_indices) == 0:
        pred_cb = global_mean
    else:
        e_target = embeddings[emb_target_idx]
        e_rated = embeddings[valid_emb_indices]
        sims = np.dot(e_rated, e_target)

        positive_mask = sims > 0
        if np.sum(positive_mask) == 0:
            pred_cb = np.mean(valid_ratings) if len(valid_ratings) > 0 else global_mean
        else:
            valid_sims = sims[positive_mask]
            v_ratings = np.array(valid_ratings)[positive_mask]
            pred_cb = np.dot(valid_sims, v_ratings) / np.sum(valid_sims)

    return np.clip(pred_cb, 0.5, 5.0)

global_mean_5star = np.mean(train_ratings_data)

# CB Validation
true_ratings_val_cb, cb_preds_val = [], []
for uid, iid, true_r in val_set:
    true_ratings_val_cb.append(true_r)
    cb_preds_val.append(get_cb_prediction(uid, iid, R_train_csr, global_mean_5star))
cb_val_rmse = math.sqrt(mean_squared_error(true_ratings_val_cb, cb_preds_val))

# CB Test
true_ratings_test_cb, cb_preds_test = [], []
for uid, iid, true_r in test_set:
    true_ratings_test_cb.append(true_r)
    cb_preds_test.append(get_cb_prediction(uid, iid, R_train_csr, global_mean_5star))
cb_test_rmse = math.sqrt(mean_squared_error(true_ratings_test_cb, cb_preds_test))

print(f"CB Validation RMSE: {cb_val_rmse:.4f}")
print(f"CB Test RMSE      : {cb_test_rmse:.4f}")

# --- BƯỚC 7: TUNE ALPHA TRÊN VALIDATION ---
print("\nĐang tối ưu hóa alpha cho Hybrid trên Validation Set...")
cf_preds_val_for_alpha = [pred.est for pred in val_predictions_cf]

alphas = np.arange(0.0, 1.01, 0.05)
best_alpha = 0.0
best_hybrid_val_rmse = float('inf')

for alpha in alphas:
    hybrid_val_preds = [alpha * cf + (1 - alpha) * cb for cf, cb in zip(cf_preds_val_for_alpha, cb_preds_val)]
    hybrid_rmse = math.sqrt(mean_squared_error(true_ratings_val_cb, hybrid_val_preds))
    if hybrid_rmse < best_hybrid_val_rmse:
        best_hybrid_val_rmse = hybrid_rmse
        best_alpha = alpha

print(f"Best alpha (Validation): {best_alpha:.2f}")
print(f"Hybrid Validation RMSE : {best_hybrid_val_rmse:.4f}")

# --- BƯỚC 8: ĐÁNH GIÁ CUỐI CÙNG TRÊN TEST ---
cf_preds_test_for_hybrid = [pred.est for pred in test_predictions_cf]
hybrid_test_preds = [best_alpha * cf + (1 - best_alpha) * cb for cf, cb in zip(cf_preds_test_for_hybrid, cb_preds_test)]
hybrid_test_rmse = math.sqrt(mean_squared_error(true_ratings_test_cb, hybrid_test_preds))

print("\n=== KẾT QUẢ ĐÁNH GIÁ RMSE CUỐI CÙNG ===")
print(f"CF Train RMSE          : {cf_train_rmse:.4f}")
print(f"CF Validation RMSE     : {cf_val_rmse:.4f}")
print(f"CF Test RMSE           : {cf_test_rmse:.4f}")
print(f"CB Validation RMSE     : {cb_val_rmse:.4f}")
print(f"CB Test RMSE           : {cb_test_rmse:.4f}")
print(f"Hybrid Validation RMSE : {best_hybrid_val_rmse:.4f} (alpha={best_alpha:.2f})")
print(f"Hybrid Test RMSE       : {hybrid_test_rmse:.4f} (alpha={best_alpha:.2f})")
print("Best SVD Params        :", best_svd_params)

rmse_5_df = pd.DataFrame([
    {'Model': 'CF (Funk SVD) Train', 'RMSE': cf_train_rmse},
    {'Model': 'CF (Funk SVD) Val', 'RMSE': cf_val_rmse},
    {'Model': 'CF (Funk SVD) Test', 'RMSE': cf_test_rmse},
    {'Model': 'Content-Based Val', 'RMSE': cb_val_rmse},
    {'Model': 'Content-Based Test', 'RMSE': cb_test_rmse},
    {'Model': f'Hybrid (a={best_alpha:.2f}) Val', 'RMSE': best_hybrid_val_rmse},
    {'Model': f'Hybrid (a={best_alpha:.2f}) Test', 'RMSE': hybrid_test_rmse}
])

display(rmse_5_df)


Tổng số ratings: 309003
Train set size: 247203
Validation set size: 30900
Test set size: 30900

Đang xây dựng R_train_csr từ tập trainset để tránh data leakage...

Đang tìm kiếm siêu tham số tối ưu cho Funk SVD trên tập trainset...
Bộ siêu tham số tốt nhất: {'n_factors': 16, 'n_epochs': 20, 'lr_all': 0.005, 'reg_all': 0.05}

Đang đánh giá hiệu suất CF (Funk SVD)...
CF Train RMSE     : 0.7620
CF Validation RMSE: 0.8969
CF Test RMSE      : 0.8873

Đang đánh giá hiệu suất CB (chỉ dùng R_train_csr)...
CB Validation RMSE: 0.9707
CB Test RMSE      : 0.9676

Đang tối ưu hóa alpha cho Hybrid trên Validation Set...
Best alpha (Validation): 0.75
Hybrid Validation RMSE : 0.8861

=== KẾT QUẢ ĐÁNH GIÁ RMSE CUỐI CÙNG ===
CF Train RMSE          : 0.7620
CF Validation RMSE     : 0.8969
CF Test RMSE           : 0.8873
CB Validation RMSE     : 0.9707
CB Test RMSE           : 0.9676
Hybrid Validation RMSE : 0.8861 (alpha=0.75)
Hybrid Test RMSE       : 0.8782 (alpha=0.75)
Best SVD Params        : {'n_fact

,Model,RMSE
0,CF (Funk SVD) Train,0.762045
1,CF (Funk SVD) Val,0.896856
2,CF (Funk SVD) Test,0.887258
3,Content-Based Val,0.970722
4,Content-Based Test,0.967639
5,Hybrid (a=0.75) Val,0.886142
6,Hybrid (a=0.75) Test,0.878181


In [ ]:
from surprise import accuracy

train_predictions = cf_algo.test(trainset.build_testset())
test_predictions = cf_algo.test(testset)

print("Train RMSE:")
accuracy.rmse(train_predictions)

print("Test RMSE:")
accuracy.rmse(test_predictions)

NameError: name 'trainset' is not defined

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))
# Sử dụng hue thay cho palette như cảnh báo trước đó để code tương thích bản mới
ax = sns.barplot(x='Model', y='RMSE', hue='Model', data=rmse_5_df, palette='magma', legend=False)

plt.title('So sánh RMSE thang 5 sao (Đã khắc phục lỗi thuật toán SVD)', fontsize=14)
plt.ylabel('Root Mean Square Error (RMSE)', fontsize=12)
plt.xlabel('Mô hình', fontsize=12)

# Thêm headroom trục y
plt.ylim(0, rmse_5_df['RMSE'].max() * 1.25)

# Gắn giá trị lên cột
for p in ax.patches:
    ax.annotate(f'{p.get_height():.4f}',
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center',
                xytext=(0, 9),
                textcoords='offset points',
                fontsize=11,
                fontweight='bold')

# Kẻ một đường baseline ở mức 1.0 (RMSE tốt cho thang 5)
plt.axhline(y=1.0, color='r', linestyle='--', alpha=0.5, label='Mức RSME tốt lý tưởng (1.0)')
plt.legend()

plt.tight_layout()
plt.show()

## 9. Trích xuất dữ liệu mẫu (Sample Data) ra CSV
Trích xuất một phần dữ liệu (100 dòng đầu) từ Input và Output của mô hình để dễ dàng kiểm tra trực quan.

In [ ]:
# Tạo thư mục chứa dữ liệu mẫu
SAMPLE_DIR = OUT_DIR / 'samples_csv'
SAMPLE_DIR.mkdir(parents=True, exist_ok=True)

print("Đang trích xuất dữ liệu mẫu ra CSV...")

# 1. places_meta (Input CB & Metadata chung)
places_meta.head(100).to_csv(SAMPLE_DIR / 'sample_places_meta.csv', index=False, encoding='utf-8-sig')

# --- THÊM MỚI: Input của CF (Rating Matrix) ---
# Lấy 100 đánh giá đầu tiên từ ma trận thưa R
R_coo = R.tocoo()
cf_input_df = pd.DataFrame({
    'UserID': cf_user_ids[R_coo.row[:100]],
    'ResID': cf_item_ids[R_coo.col[:100]],
    'Rating': R_coo.data[:100]
})
cf_input_df.to_csv(SAMPLE_DIR / 'sample_cf_input_ratings.csv', index=False, encoding='utf-8-sig')

# 2. Content-Based Lookup (Output CB)
# Trích xuất 100 items đầu tiên
cb_sample = {k: cb_lookup[k] for k in list(cb_lookup.keys())[:100]}
cb_df = pd.DataFrame([{"ResID": k, "Similar_Items": v} for k, v in cb_sample.items()])
cb_df.to_csv(SAMPLE_DIR / 'sample_cb_lookup.csv', index=False, encoding='utf-8-sig')

# 3. Collaborative Filtering Lookup (Output CF)
# Trích xuất 100 users đầu tiên
cf_sample = {k: cf_lookup.get(k, []) for k in list(cf_lookup.keys())[:100]}
cf_df = pd.DataFrame([{"UserID": k, "Recommended_Items": v} for k, v in cf_sample.items()])
cf_df.to_csv(SAMPLE_DIR / 'sample_cf_lookup.csv', index=False, encoding='utf-8-sig')

# 4. User & Item Factors (Output SVD)
# Trích xuất 100 vectors đầu tiên
pd.DataFrame(P[:100]).to_csv(SAMPLE_DIR / 'sample_user_factors_P.csv', index=False)
pd.DataFrame(Q[:100]).to_csv(SAMPLE_DIR / 'sample_item_factors_Q.csv', index=False)

# 5. Content Embeddings (Output Embedding)
# Trích xuất 100 vectors đầu tiên của ma trận embeddings
pd.DataFrame(embeddings[:100]).to_csv(SAMPLE_DIR / 'sample_content_embeddings.csv', index=False)

print(f"Đã lưu thành công các file CSV mẫu tại: {SAMPLE_DIR}")

# Hiển thị thử dữ liệu Input của CF
display(cf_input_df.head())